# EISForge → AutoREC disk workflow

This small workflow lets EISForge export one synthetic sample, then loads that sample through AutoREC's existing disk-based input path. A temporary directory keeps generated files out of the repository.

In [ ]:
# Configure native scientific libraries before importing AutoEIS-dependent modules.
from autorec.runtime import configure_autorec_runtime

configure_autorec_runtime(thread_count=1, warmup_autoeis=True, suppress_tf_logs=True)

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from eisforge import DataGen
from autorec.data_preparation import EISDataPrep

In [ ]:
# Use a temporary output directory so running the demo never dirties the repository.
with TemporaryDirectory(prefix="eisforge-autorec-") as temp_dir:
    output_dir = Path(temp_dir)
    generator = DataGen(
        random_ecm_circuit="R1-[P2,R3]",
        output_dir=output_dir,
        n_random_candidates=20,
        max_selected_curves=3,
        fim_fit_ecm=False,
        drop_pp_series=False,
        drop_invalid_ecms=False,
        excluded_simplified_ecms=(),
        verbose=False,
    )
    generated_rows, batch_details = generator.generate_data(
        target_num=1,
        max_batches=6,
        seed_start=7,
        export_summary=False,
        export_samples=True,
        live_plot=False,
    )

    sample_files = sorted(output_dir.rglob("sample_*.csv"))
    assert len(sample_files) == len(generated_rows) == 1

    # AutoREC reads the files exactly as it would read a saved dataset.
    autorec_dataset = EISDataPrep(
        output_dir, mode="process", evaluation=True
    ).load()

assert not output_dir.exists()
assert len(autorec_dataset) == 1
assert set(EISDataPrep.EVAL_REQUIRED_COLUMNS).issubset(autorec_dataset.columns)
autorec_dataset[["sub_id", "true_circuit", "chi_thresh", "r2_thresh"]]